# CSE428 Project — Pet Segmentation & Breed Classification

**Dataset:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) · 37 breeds · 3,680 trainval / 3,669 test images

**Task:** jointly segment the pet (binary mask) **and** classify the breed (37 classes).

**Architectures:**
1. **U-Net** (Ronneberger et al., 2015)
2. **Attention U-Net** (Oktay et al., 2018)

Each has a classifier head on the encoder bottleneck, trained jointly with
`L = L_seg + λ·L_cls`.

**This notebook is the whole project.** Set `MODE` in the config cell below:

- `MODE = "train"` — trains both models from scratch (needs the dataset + GPU),
  saving `data/unet/` and `data/attention_unet/` artifacts.
- `MODE = "present"` — loads the saved artifacts and presents the results
  (no training). This is what you run for the demonstration.


## 0. Configuration

Choose the mode, then run the cell below.


In [ ]:
import torch

# ============ CONFIG ============
MODE = "present"          # "train"  -> train both models from scratch
                          # "present" -> load saved artifacts and present

# Dataset location (auto-downloaded in train mode if missing)
DATA_DIR = "data"

# Output directory for artifacts (train mode) / where to look (present mode)
OUT_DIR = "outputs"

# --- training hyperparameters (train mode) ---
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS_UNET = 80          # set lower (e.g. 2) to smoke-test on CPU
EPOCHS_ATTN = 80
LR = 3.0e-4
WEIGHT_DECAY = 1.0e-4
LAMBDA_CLS = 1.0
SEED = 42
NUM_WORKERS = 2
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
)

print(f"mode: {MODE} | device: {DEVICE}")


In [ ]:
import os, random, glob, json, time, shutil
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# --- dataset ---
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import ColorJitter
from torchvision.transforms import functional as TF

def download_dataset():
    """Download Oxford-IIIT Pet into DATA_DIR if not present."""
    os.makedirs(DATA_DIR, exist_ok=True)
    # torchvision downloads into DATA_DIR/oxford-iiit-pet
    OxfordIIITPet(root=DATA_DIR, split="trainval", target_types=("category","segmentation"), download=True)
    OxfordIIITPet(root=DATA_DIR, split="test", target_types=("category","segmentation"), download=True)
    print("dataset ready under", os.path.join(DATA_DIR, "oxford-iiit-pet"))

if MODE == "train":
    if not os.path.isdir(os.path.join(DATA_DIR, "oxford-iiit-pet", "images")):
        download_dataset()
    else:
        print("dataset already present")


## Dataset & loaders (train mode) / present-mode loader

In [ ]:
import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import ColorJitter
from torchvision.transforms import functional as TF

NUM_CLASSES = 37

KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/oxford-iiit-pet",
    "/kaggle/input/oxford-iiit-pets",
]


def resolve_data_root(configured=None) -> str:
    if configured:
        return configured
    env = os.environ.get("CSE428_DATA_ROOT")
    if env and os.path.isdir(env):
        return env
    for cand in KAGGLE_DATA_CANDIDATES:
        if os.path.isdir(os.path.join(cand, "oxford-iiit-pet")):
            return cand
        if os.path.isdir(os.path.join(cand, "images")) and os.path.isdir(
            os.path.join(cand, "annotations")
        ):
            return os.path.dirname(cand)
    return "./data"


def make_splits(n_total: int, val_frac: float = 0.1, seed: int = 42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n_total)
    n_val = int(round(n_total * val_frac))
    return perm[n_val:], perm[:n_val]


class PetSegDataset(Dataset):

    def __init__(
        self,
        root,
        split="trainval",
        indices=None,
        img_size=256,
        augment=False,
        download=False,
        return_trimap=False,
    ):
        self.base = OxfordIIITPet(
            root=root,
            split=split,
            target_types=("category", "segmentation"),
            download=download,
        )
        self.split = split
        self.indices = (
            np.arange(len(self.base)) if indices is None else np.asarray(indices)
        )
        self.img_size = img_size
        self.augment = augment
        self.return_trimap = return_trimap
        self.classes = list(self.base.classes)
        self.jitter = ColorJitter(
            brightness=0.3, contrast=0.3, saturation=0.3, hue=0.06
        )

    def label_name(self, idx: int) -> str:
        return self.classes[int(idx)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, (label, trimap) = self.base[int(self.indices[i])]
        img = TF.resize(img, [self.img_size, self.img_size], antialias=True)
        trimap = TF.resize(
            trimap,
            [self.img_size, self.img_size],
            interpolation=TF.InterpolationMode.NEAREST,
        )
        if self.augment:
            img, trimap = self._augment(img, trimap)
        mask = (np.array(trimap, dtype=np.uint8) != 2).astype(np.int64)
        sample = {
            "image": TF.to_tensor(img),
            "mask": torch.from_numpy(mask),
            "label": torch.tensor(int(label), dtype=torch.long),
        }
        if self.return_trimap:
            sample["trimap"] = torch.from_numpy(np.array(trimap, dtype=np.int64))
        return sample

    def _augment(self, img, trimap):
        interp = TF.InterpolationMode
        # random rotation (-15..15 deg), pad with background class so corners
        # stay consistent with the raw trimap convention
        if torch.rand(()) < 0.5:
            angle = float(torch.distributions.Uniform(-15.0, 15.0).sample(()))
            img = TF.rotate(img, angle, interpolation=interp.BILINEAR, fill=0)
            trimap = TF.rotate(
                trimap, angle, interpolation=interp.NEAREST, fill=2
            )
        # random resized crop: crop a box of 80-100% area, resize back to img_size
        if torch.rand(()) < 0.5:
            scale = float(torch.distributions.Uniform(0.8, 1.0).sample(()))
            box = max(2, int(round(self.img_size * scale)))
            top = int(torch.rand([]).item() * (self.img_size - box))
            left = int(torch.rand([]).item() * (self.img_size - box))
            img = TF.resized_crop(
                img, top, left, box, box, [self.img_size, self.img_size],
                interpolation=interp.BILINEAR, antialias=True,
            )
            trimap = TF.resized_crop(
                trimap, top, left, box, box, [self.img_size, self.img_size],
                interpolation=interp.NEAREST,
            )
        # horizontal flip
        if torch.rand(()) < 0.5:
            img = TF.hflip(img)
            trimap = TF.hflip(trimap)
        # color jitter (image only - the mask has no color)
        img = self.jitter(img)
        return img, trimap


def get_datasets(
    root=None,
    img_size=256,
    val_frac=0.1,
    seed=42,
    augment=True,
    download=False,
    return_trimap=False,
):
    root = resolve_data_root(root)
    probe = PetSegDataset(root, split="trainval", img_size=img_size, download=download)
    train_idx, val_idx = make_splits(len(probe), val_frac=val_frac, seed=seed)
    train_ds = PetSegDataset(
        root, "trainval", indices=train_idx, img_size=img_size,
        augment=augment, return_trimap=return_trimap,
    )
    val_ds = PetSegDataset(
        root, "trainval", indices=val_idx, img_size=img_size,
        augment=False, return_trimap=return_trimap,
    )
    test_ds = PetSegDataset(
        root, "test", img_size=img_size, augment=False, return_trimap=return_trimap
    )
    return train_ds, val_ds, test_ds


def get_loaders(
    root=None,
    img_size=256,
    val_frac=0.1,
    seed=42,
    augment=True,
    batch_size=16,
    num_workers=2,
    download=False,
    return_trimap=False,
):
    train_ds, val_ds, test_ds = get_datasets(
        root=root,
        img_size=img_size,
        val_frac=val_frac,
        seed=seed,
        augment=augment,
        download=download,
        return_trimap=return_trimap,
    )
    common = dict(num_workers=num_workers, pin_memory=torch.cuda.is_available())
    datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
    loaders = {
        "train": DataLoader(
            train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **common
        ),
        "val": DataLoader(val_ds, batch_size=batch_size, shuffle=False, **common),
        "test": DataLoader(test_ds, batch_size=batch_size, shuffle=False, **common),
    }
    return datasets, loaders

## Model definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):

    def __init__(self, in_ch=3, base_channels=32, num_classes=37):
        super().__init__()
        c = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.enc1 = DoubleConv(in_ch, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[3] * 2)

        self.up4 = nn.ConvTranspose2d(c[3] * 2, c[3], 2, stride=2)
        self.dec4 = DoubleConv(c[3] * 2, c[3])
        self.up3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
        self.dec3 = DoubleConv(c[2] * 2, c[2])
        self.up2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
        self.dec2 = DoubleConv(c[1] * 2, c[1])
        self.up1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
        self.dec1 = DoubleConv(c[0] * 2, c[0])

        self.seg_head = nn.Conv2d(c[0], 1, 1)
        self.cls_head = nn.Sequential(
            nn.Linear(c[3] * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def encode(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        b = self.bottleneck(self.pool(s4))
        return (s1, s2, s3, s4), b

    def decode(self, b, skips):
        s1, s2, s3, s4 = skips
        x = self.dec4(torch.cat([self.up4(b), s4], dim=1))
        x = self.dec3(torch.cat([self.up3(x), s3], dim=1))
        x = self.dec2(torch.cat([self.up2(x), s2], dim=1))
        x = self.dec1(torch.cat([self.up1(x), s1], dim=1))
        return x

    def forward(self, x):
        skips, b = self.encode(x)
        seg_logits = self.seg_head(self.decode(b, skips))
        cls_logits = self.cls_head(F.adaptive_avg_pool2d(b, 1).flatten(1))
        return {"seg_logits": seg_logits, "cls_logits": cls_logits, "bottleneck": b}


import torch
import torch.nn as nn
import torch.nn.functional as F



class AttentionGate(nn.Module):

    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.w_g = nn.Sequential(
            nn.Conv2d(gate_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch)
        )
        self.w_x = nn.Sequential(
            nn.Conv2d(skip_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # The gating signal g is coarser than the skip feature x; upsample it to
        # x's spatial resolution first so the two 1x1 projections can be added.
        g = F.interpolate(
            g, size=x.shape[-2:], mode="bilinear", align_corners=False
        )
        a = self.relu(self.w_g(g) + self.w_x(x))
        return x * self.psi(a)


class AttentionUNet(UNet):

    def __init__(self, in_ch=3, base_channels=32, num_classes=37):
        super().__init__(in_ch=in_ch, base_channels=base_channels, num_classes=num_classes)
        c = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.a4 = AttentionGate(c[3] * 2, c[3], c[3] // 2)
        self.a3 = AttentionGate(c[3], c[2], c[2] // 2)
        self.a2 = AttentionGate(c[2], c[1], c[1] // 2)
        self.a1 = AttentionGate(c[1], c[0], c[0] // 2)

    def forward(self, x):
        skips, b = self.encode(x)
        s1, s2, s3, s4 = skips
        d = self.dec4(torch.cat([self.up4(b), self.a4(b, s4)], dim=1))
        d = self.dec3(torch.cat([self.up3(d), self.a3(d, s3)], dim=1))
        d = self.dec2(torch.cat([self.up2(d), self.a2(d, s2)], dim=1))
        d = self.dec1(torch.cat([self.up1(d), self.a1(d, s1)], dim=1))
        seg_logits = self.seg_head(d)
        cls_logits = self.cls_head(F.adaptive_avg_pool2d(b, 1).flatten(1))
        return {"seg_logits": seg_logits, "cls_logits": cls_logits, "bottleneck": b}


__all__ = ["UNet", "AttentionUNet", "build_model", "build_backbone_classifier"]


def build_model(cfg):
    name = cfg["model"]["name"]
    kwargs = dict(
        base_channels=cfg["model"].get("base_channels", 32),
        num_classes=cfg["model"].get("num_classes", NUM_CLASSES),
    )
    if name == "unet":
        return UNet(**kwargs)
    if name == "attention_unet":
        return AttentionUNet(**kwargs)
    raise ValueError(f"unknown model: {name}")


# ---------------------------------------------------------------------------
# Local model factory (self-contained equivalent of the project factory)
# ---------------------------------------------------------------------------

NUM_CLASSES = 37


def build_model(cfg):
    name = cfg["model"]["name"]
    kwargs = dict(
        base_channels=cfg["model"].get("base_channels", 32),
        num_classes=cfg["model"].get("num_classes", NUM_CLASSES),
    )
    if name == "unet":
        return UNet(**kwargs)
    if name == "attention_unet":
        return AttentionUNet(**kwargs)
    raise ValueError(f"unknown model: {name}")


## Metrics

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


class SegMetricAccumulator:

    def __init__(self, num_classes=2):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.inter = torch.zeros(self.num_classes, dtype=torch.float64)
        self.union = torch.zeros(self.num_classes, dtype=torch.float64)
        self.correct = 0.0
        self.total = 0.0

    @torch.no_grad()
    def update(self, preds, targets):
        preds = preds.detach().cpu().view(-1)
        targets = targets.detach().cpu().view(-1)
        for c in range(self.num_classes):
            p, t = preds == c, targets == c
            self.inter[c] += (p & t).sum().item()
            self.union[c] += (p | t).sum().item()
        self.correct += (preds == targets).sum().item()
        self.total += preds.numel()

    def compute(self):
        iou = self.inter / self.union.clamp(min=1.0)
        dice = 2.0 * self.inter / (self.inter + self.union).clamp(min=1.0)
        present = self.union > 0
        if present.any():
            miou = iou[present].mean().item()
            mdice = dice[present].mean().item()
        else:
            miou = mdice = 1.0
        return {
            "miou": miou,
            "dice": mdice,
            "iou_fg": iou[1].item(),
            "dice_fg": dice[1].item(),
            "pixel_acc": self.correct / max(self.total, 1.0),
        }


class ClsMetricAccumulator:

    def __init__(self):
        self.reset()

    def reset(self):
        self.preds = []
        self.targets = []

    @torch.no_grad()
    def update(self, logits, targets):
        self.preds.extend(logits.argmax(1).detach().cpu().tolist())
        self.targets.extend(targets.detach().cpu().tolist())

    def compute(self):
        p, t = np.asarray(self.preds), np.asarray(self.targets)
        precision, recall, f1, _ = precision_recall_fscore_support(
            t, p, average="macro", zero_division=0
        )
        return {
            "acc": accuracy_score(t, p),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }


@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    seg_acc = SegMetricAccumulator()
    cls_acc = ClsMetricAccumulator()
    for batch in loader:
        out = model(batch["image"].to(device))
        if not isinstance(out, dict):
            out = {"cls_logits": out}
        cls_acc.update(out["cls_logits"], batch["label"])
        seg_logits = out.get("seg_logits")
        if seg_logits is not None:
            preds = (torch.sigmoid(seg_logits) > threshold).long().squeeze(1)
            seg_acc.update(preds, batch["mask"])
    return seg_acc.compute(), cls_acc.compute()

## Training code

In [ ]:
import glob
import json
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast



class DiceBCELoss(nn.Module):

    def __init__(self, eps=1.0):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits)
        dims = tuple(range(1, logits.dim()))
        inter = (probs * targets).sum(dim=dims)
        cardinality = probs.sum(dim=dims) + targets.sum(dim=dims)
        dice = 1.0 - ((2.0 * inter + self.eps) / (cardinality + self.eps)).mean()
        return bce + dice


def find_resume_checkpoint(explicit=None, out_dir=None, model_name=None):
    candidates = []
    if out_dir:
        candidates.append(os.path.join(out_dir, "checkpoints", "last.pth"))
    if explicit:
        candidates.append(explicit)
    if os.path.isdir("/kaggle/input"):
        candidates += sorted(
            glob.glob("/kaggle/input/**/checkpoints/last.pth", recursive=True)
        )
        candidates += sorted(
            glob.glob("/kaggle/input/**/outputs/*/checkpoints/last.pth", recursive=True)
        )
    for path in candidates:
        if not os.path.exists(path):
            continue
        try:
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
        except Exception:
            continue
        if (
            model_name
            and ckpt.get("cfg", {}).get("model", {}).get("name") != model_name
        ):
            continue
        return path
    return None


class Trainer:

    def __init__(self, model, loaders, device, cfg, out_dir, resume=None):
        self.model = model.to(device)
        self.loaders = loaders
        self.classes = (
            list(loaders["train"].dataset.classes)
            if hasattr(loaders["train"].dataset, "classes")
            else None
        )
        self.device = device
        self.cfg = cfg
        self.out_dir = out_dir
        self.ckpt_dir = os.path.join(out_dir, "checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)

        train_cfg, model_cfg = cfg["train"], cfg["model"]
        self.epochs_total = train_cfg["epochs_total"]
        self.lambda_cls = model_cfg.get("lambda_cls", 1.0)
        self.amp = train_cfg.get("amp", True) and device.type == "cuda"

        self.optim = torch.optim.AdamW(
            self.model.parameters(),
            lr=train_cfg["lr"],
            weight_decay=train_cfg.get("weight_decay", 0.0),
        )
        self.scaler = GradScaler("cuda", enabled=self.amp)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optim, T_max=self.epochs_total
        )
        self.seg_criterion = DiceBCELoss()
        self.cls_criterion = nn.CrossEntropyLoss(
            label_smoothing=train_cfg.get("label_smoothing", 0.1)
        )

        self.history = []
        self.start_epoch = 1
        self.best_miou = -1.0
        if resume:
            self._load_resume(resume)

    def _load_resume(self, path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        try:
            self.model.load_state_dict(ckpt["model_state"])
            self.optim.load_state_dict(ckpt["optim_state"])
            self.scheduler.load_state_dict(ckpt["scheduler_state"])
        except RuntimeError as e:
            # Architecture may differ (e.g. improved classifier head) between
            # versions -> keep the optimizer/scheduler but start epochs fresh.
            print("WARNING: checkpoint architecture mismatch "
                  f"({type(e).__name__}); will train from epoch 1.")
            self.model = build_model(self.cfg).to(self.device)
            return
        self.scheduler.T_max = self.epochs_total
        self.history = ckpt.get("history", [])
        self.start_epoch = ckpt["epoch"] + 1
        self.best_miou = ckpt.get("best_miou", -1.0)
        print(
            f"resumed from {path} at epoch {ckpt['epoch']} "
            f"(best val mIoU {self.best_miou:.4f}); "
            f"training epochs {self.start_epoch}..{self.epochs_total}"
        )

    def _train_epoch(self):
        self.model.train()
        totals = {"loss": 0.0, "seg": 0.0, "cls": 0.0}
        seen = 0
        for batch in self.loaders["train"]:
            imgs = batch["image"].to(self.device, non_blocking=True)
            masks = batch["mask"].unsqueeze(1).to(self.device, non_blocking=True)
            labels = batch["label"].to(self.device, non_blocking=True)

            self.optim.zero_grad(set_to_none=True)
            with autocast(self.device.type, enabled=self.amp):
                out = self.model(imgs)
                if not isinstance(out, dict):
                    out = {"cls_logits": out}
                if out.get("seg_logits") is not None:
                    seg_loss = self.seg_criterion(out["seg_logits"], masks)
                else:
                    seg_loss = torch.zeros((), device=self.device)
                cls_loss = self.cls_criterion(out["cls_logits"], labels)
                loss = seg_loss + self.lambda_cls * cls_loss

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optim)
            self.scaler.update()

            bs = imgs.size(0)
            totals["loss"] += loss.item() * bs
            totals["seg"] += seg_loss.item() * bs
            totals["cls"] += cls_loss.item() * bs
            seen += bs

        self.scheduler.step()
        lr = self.optim.param_groups[0]["lr"]
        return {
            "train_loss": totals["loss"] / seen,
            "train_seg_loss": totals["seg"] / seen,
            "train_cls_loss": totals["cls"] / seen,
            "lr": lr,
        }

    def _snapshot(self):
        return {
            "epoch": self.history[-1]["epoch"] if self.history else 0,
            "model_state": self.model.state_dict(),
            "optim_state": self.optim.state_dict(),
            "scheduler_state": self.scheduler.state_dict(),
            "history": self.history,
            "best_miou": self.best_miou,
            "cfg": self.cfg,
            "classes": self.classes,
        }

    def fit(self):
        if self.start_epoch > self.epochs_total:
            print(
                f"all {self.epochs_total} epochs already complete - "
                "bump train.epochs_total to train further"
            )
            return []
        new_records = []
        for epoch in range(self.start_epoch, self.epochs_total + 1):
            t0 = time.time()
            train_stats = self._train_epoch()
            seg_m, cls_m = evaluate(self.model, self.loaders["val"], self.device)
            rec = {"epoch": epoch, "time_s": round(time.time() - t0, 1), **train_stats}
            rec.update({f"val_{k}": v for k, v in seg_m.items()})
            rec.update({f"val_{k}": v for k, v in cls_m.items()})
            self.history.append(rec)
            new_records.append(rec)

            is_best = seg_m["miou"] > self.best_miou
            if is_best:
                self.best_miou = seg_m["miou"]
            torch.save(self._snapshot(), os.path.join(self.ckpt_dir, "last.pth"))
            if is_best:
                torch.save(self._snapshot(), os.path.join(self.ckpt_dir, "best.pth"))

            print(
                f"epoch {epoch}/{self.epochs_total} | "
                f"loss {rec['train_loss']:.4f} "
                f"(seg {rec['train_seg_loss']:.4f} cls {rec['train_cls_loss']:.4f}) | "
                f"val mIoU {seg_m['miou']:.4f} dice {seg_m['dice']:.4f} "
                f"pacc {seg_m['pixel_acc']:.4f} | "
                f"acc {cls_m['acc']:.4f} f1 {cls_m['f1']:.4f} | "
                f"{rec['time_s']:.1f}s"
                + (" *best*" if is_best else "")
            )
        with open(os.path.join(self.out_dir, "history.json"), "w") as f:
            json.dump(self.history, f, indent=2)
        self.start_epoch = self.epochs_total + 1
        return new_records

    def final_report(self):
        best_path = os.path.join(self.ckpt_dir, "best.pth")
        if os.path.exists(best_path):
            try:
                ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
                self.model.load_state_dict(ckpt["model_state"])
                print(
                    f"final report with best checkpoint: epoch {ckpt['epoch']}, "
                    f"val mIoU {ckpt['best_miou']:.4f}"
                )
            except RuntimeError as e:
                print("WARNING: best.pth architecture mismatch, using current model.", e)
        results = {}
        for name in ("train", "val", "test"):
            seg_m, cls_m = evaluate(self.model, self.loaders[name], self.device)
            results[name] = {"seg": seg_m, "cls": cls_m}
            print(
                f"{name:>5}: mIoU {seg_m['miou']:.4f} dice {seg_m['dice']:.4f} "
                f"pacc {seg_m['pixel_acc']:.4f} | "
                f"acc {cls_m['acc']:.4f} prec {cls_m['precision']:.4f} "
                f"rec {cls_m['recall']:.4f} f1 {cls_m['f1']:.4f}"
            )
        with open(os.path.join(self.out_dir, "results.json"), "w") as f:
            json.dump(results, f, indent=2)
        return results

## 1. Train (MODE='train') or load (MODE='present')

In **train** mode, both models are trained and the artifacts are saved under
`data/`. In **present** mode, the saved artifacts are loaded.


In [ ]:
# Artifact discovery (used in both modes: present loads, train saves then loads)
def load_results(name):
    p = os.path.join(DATA_DIR, name, "results.json")
    return json.load(open(p)) if os.path.exists(p) else None

def load_history(name):
    p = os.path.join(DATA_DIR, name, "history.json")
    return json.load(open(p)) if os.path.exists(p) else None

def find_best(name):
    return glob.glob(os.path.join(DATA_DIR, name, "checkpoints", "best.pth"))

def report_artifacts():
    for n in ("unet", "attention_unet"):
        r, h, b = load_results(n), load_history(n), find_best(n)
        print(f"{n:>15}: results={'ok' if r else 'MISSING':7} "
              f"history={'ok' if h else 'MISSING':7} best.pth={'ok' if b else 'MISSING'}")

# Build the datasets + loaders (train mode only)
def make_loaders():
    from torch.utils.data import DataLoader
    ds_train = PetSegDataset(DATA_DIR, split="trainval", img_size=IMG_SIZE, augment=True,
                             download=(MODE == "train"))
    rng = np.random.RandomState(SEED)
    perm = rng.permutation(len(ds_train))
    n_val = int(round(len(ds_train) * 0.1))
    train_idx, val_idx = perm[n_val:], perm[:n_val]
    train_ds = PetSegDataset(DATA_DIR, split="trainval", indices=train_idx, img_size=IMG_SIZE,
                             augment=True, download=(MODE == "train"))
    val_ds = PetSegDataset(DATA_DIR, split="trainval", indices=val_idx, img_size=IMG_SIZE,
                           augment=False, download=(MODE == "train"))
    test_ds = PetSegDataset(DATA_DIR, split="test", img_size=IMG_SIZE, augment=False,
                            download=(MODE == "train"))
    common = dict(num_workers=0, pin_memory=False)
    loaders = {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **common),
        "val": DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **common),
        "test": DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **common),
    }
    return {"train": train_ds, "val": val_ds, "test": test_ds}, loaders

if MODE == "train":
    datasets, loaders = make_loaders()
    print("train", len(datasets["train"]), "val", len(datasets["val"]), "test", len(datasets["test"]))
else:
    report_artifacts()


## 2. Train / load U-Net


In [ ]:
if MODE == "train":
    cfg = {
        "model": {"name": "unet", "base_channels": 32, "num_classes": 37, "lambda_cls": LAMBDA_CLS},
        "train": {"lr": LR, "weight_decay": WEIGHT_DECAY, "epochs_total": EPOCHS_UNET, "label_smoothing": 0.1},
        "data": {"img_size": IMG_SIZE},
    }
    unet_out = os.path.join(OUT_DIR, "unet")
    model = build_model(cfg).to(DEVICE)
    tr = Trainer(model, loaders, DEVICE, cfg, out_dir=unet_out)
    tr.fit()
    tr.final_report()
    # copy artifacts to DATA_DIR for the present-mode layout
    os.makedirs(os.path.join(DATA_DIR, "unet", "checkpoints"), exist_ok=True)
    shutil.copy(os.path.join(unet_out, "results.json"), os.path.join(DATA_DIR, "unet", "results.json"))
    shutil.copy(os.path.join(unet_out, "history.json"), os.path.join(DATA_DIR, "unet", "history.json"))
    shutil.copy(os.path.join(unet_out, "checkpoints", "best.pth"), os.path.join(DATA_DIR, "unet", "checkpoints", "best.pth"))
    shutil.copy(os.path.join(unet_out, "checkpoints", "last.pth"), os.path.join(DATA_DIR, "unet", "checkpoints", "last.pth"))
    print("U-Net artifacts saved to", os.path.join(DATA_DIR, "unet"))

# refresh globals so present/demo cells work in both modes
UNET_RESULTS, UNET_HISTORY, UNET_BEST = load_results("unet"), load_history("unet"), find_best("unet")


## 3. Train / load Attention U-Net


In [ ]:
if MODE == "train":
    cfg = {
        "model": {"name": "attention_unet", "base_channels": 32, "num_classes": 37, "lambda_cls": LAMBDA_CLS},
        "train": {"lr": LR, "weight_decay": WEIGHT_DECAY, "epochs_total": EPOCHS_ATTN, "label_smoothing": 0.1},
        "data": {"img_size": IMG_SIZE},
    }
    attn_out = os.path.join(OUT_DIR, "attention_unet")
    model = build_model(cfg).to(DEVICE)
    tr = Trainer(model, loaders, DEVICE, cfg, out_dir=attn_out)
    tr.fit()
    tr.final_report()
    os.makedirs(os.path.join(DATA_DIR, "attention_unet", "checkpoints"), exist_ok=True)
    shutil.copy(os.path.join(attn_out, "results.json"), os.path.join(DATA_DIR, "attention_unet", "results.json"))
    shutil.copy(os.path.join(attn_out, "history.json"), os.path.join(DATA_DIR, "attention_unet", "history.json"))
    shutil.copy(os.path.join(attn_out, "checkpoints", "best.pth"), os.path.join(DATA_DIR, "attention_unet", "checkpoints", "best.pth"))
    shutil.copy(os.path.join(attn_out, "checkpoints", "last.pth"), os.path.join(DATA_DIR, "attention_unet", "checkpoints", "last.pth"))
    print("Attention U-Net artifacts saved to", os.path.join(DATA_DIR, "attention_unet"))

# refresh globals so present/demo cells work in both modes
ATTN_RESULTS, ATTN_HISTORY, ATTN_BEST = load_results("attention_unet"), load_history("attention_unet"), find_best("attention_unet")
report_artifacts()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
def seg_row(r):
    s = r["seg"]; return {"mIoU": s["miou"], "Dice": s["dice"], "Pixel acc": s["pixel_acc"]}
def cls_row(r):
    c = r["cls"]; return {"Acc": c["acc"], "Prec": c["precision"], "Rec": c["recall"], "F1": c["f1"]}
def results_table(results):
    rows = []
    for split in ("train", "val", "test"):
        rows.append({"Split": split, **seg_row(results[split]), **cls_row(results[split])})
    return pd.DataFrame(rows).set_index("Split").round(4)
def history_plot(history, title):
    ep = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
    axes[0].plot(ep, [h["train_loss"] for h in history], label="total")
    axes[0].plot(ep, [h["train_seg_loss"] for h in history], label="segmentation")
    axes[0].plot(ep, [h["train_cls_loss"] for h in history], label="classification")
    axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch")
    axes[1].plot(ep, [h["val_miou"] for h in history], label="mIoU")
    axes[1].plot(ep, [h["val_dice"] for h in history], label="Dice")
    axes[1].plot(ep, [h["val_pixel_acc"] for h in history], label="pixel acc")
    axes[1].set_title("Validation - segmentation"); axes[1].set_xlabel("epoch")
    axes[2].plot(ep, [h["val_acc"] for h in history], label="accuracy")
    axes[2].plot(ep, [h["val_precision"] for h in history], label="precision")
    axes[2].plot(ep, [h["val_recall"] for h in history], label="recall")
    axes[2].plot(ep, [h["val_f1"] for h in history], label="F1")
    axes[2].set_title("Validation - classification"); axes[2].set_xlabel("epoch")
    for ax in axes: ax.grid(alpha=0.3); ax.legend(fontsize=9)
    fig.suptitle(title); fig.tight_layout()
    return fig


## 4. Results — U-Net


In [ ]:
if UNET_HISTORY:
    fig = history_plot(UNET_HISTORY, "U-Net - training curves")
    plt.show()
if UNET_RESULTS:
    display(results_table(UNET_RESULTS))
else:
    print("no U-Net results found (train first, or set MODE=present with data/)")


## 5. Results — Attention U-Net


In [ ]:
if ATTN_HISTORY:
    fig = history_plot(ATTN_HISTORY, "Attention U-Net - training curves")
    plt.show()
if ATTN_RESULTS:
    display(results_table(ATTN_RESULTS))
else:
    print("no Attention U-Net results found (train first, or set MODE=present with data/)")


## 6. Comparison — U-Net vs Attention U-Net


In [ ]:
rows = {}
for name, results, best in [("unet", UNET_RESULTS, UNET_BEST),
                            ("attention_unet", ATTN_RESULTS, ATTN_BEST)]:
    if results is None or not best:
        rows[name] = {"status": "no data", "mIoU": None, "Dice": None, "Pixel acc": None,
                      "Acc": None, "F1": None, "params": None}
        continue
    ck = torch.load(best[0], map_location="cpu", weights_only=False)
    n_params = sum(p.numel() for p in build_model(ck["cfg"]).parameters())
    t = results["test"]
    rows[name] = {"status": "ok", "mIoU": t["seg"]["miou"], "Dice": t["seg"]["dice"],
                  "Pixel acc": t["seg"]["pixel_acc"], "Acc": t["cls"]["acc"],
                  "F1": t["cls"]["f1"], "params": n_params}
compare = pd.DataFrame(rows).T
display(compare.round(4))
print("\nDiscussion:")
print("- Attention gates add parameters to each skip path; both models land at a similar")
print("  mIoU (~0.89), with the classifier head reaching ~0.72 test accuracy.")


## 7. Live demonstration — instant prediction (no training)

Load a saved model and run **one forward pass** on any image (upload or path).


In [ ]:
from pathlib import Path
from IPython.display import display
try:
    import ipywidgets as widgets
    _HAVE_WIDGETS = True
except ImportError:
    widgets = None; _HAVE_WIDGETS = False

DEMO_MODEL = "attention_unet"     # "unet" | "attention_unet"
DEMO_IMG = ""                      # path to an image, or leave empty to upload

_best = ATTN_BEST if DEMO_MODEL == "attention_unet" else UNET_BEST
if not _best:
    print(f"no best.pth for {DEMO_MODEL} - copy it under data/ to run the demo.")
else:
    _ck = torch.load(_best[0], map_location="cpu", weights_only=False)
    _model = build_model(_ck["cfg"]).to(DEVICE)
    _model.load_state_dict(_ck["model_state"]); _model.eval()
    _IMGSIZE = int(_ck["cfg"].get("data", {}).get("img_size", 256))
    _CLASSES = list(_ck.get("classes") or [f"class_{i}" for i in range(37)])
    _THRESH = float(_ck["cfg"].get("model", {}).get("seg_threshold", 0.5))
    print(f"loaded {DEMO_MODEL} | img_size {_IMGSIZE} | threshold {_THRESH} | {len(_CLASSES)} breeds")

    if DEMO_IMG.strip():
        _demo_path = Path(DEMO_IMG).expanduser()
    elif _HAVE_WIDGETS:
        _up = widgets.FileUpload(accept="image/*", multiple=False, description="Upload image")
        display(_up)
        print("Drop your image into the upload button above, then run the next cell.")
    else:
        print("set DEMO_IMG to an image path (ipywidgets not installed).")


In [ ]:
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt

if "_model" not in globals():
    print("run the previous cell first (it loads the model).")
else:
    if DEMO_IMG.strip():
        _demo_path = Path(DEMO_IMG).expanduser()
    elif _HAVE_WIDGETS:
        _up = widgets.FileUpload(accept="image/*", multiple=False, description="Upload image")
        display(_up)
        _first = list(_up.value.values())[0]
        _demo_path = Path("./_uploaded.png")
        _demo_path.write_bytes(bytes(_first["content"] if isinstance(_first, dict) else _first))
    else:
        print("set DEMO_IMG to an image path (ipywidgets not installed).")
        _demo_path = None

    if _demo_path is not None:
        _img = Image.open(_demo_path).convert("RGB")
        x = TF.resize(_img, [_IMGSIZE, _IMGSIZE], antialias=True)
        x = TF.to_tensor(x).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = _model(x)
        _mask = (torch.sigmoid(out["seg_logits"]) > _THRESH).float()
        _mask = TF.resize(_mask[0, 0].unsqueeze(0).unsqueeze(0), [_img.size[1], _img.size[0]],
                          interpolation=TF.InterpolationMode.NEAREST)[0, 0].cpu().numpy()
        _probs = torch.softmax(out["cls_logits"], dim=1)[0]
        _conf, _lab = _probs.max(0)
        print(f"predicted breed: {_CLASSES[int(_lab)]}")
        print(f"confidence:      {_conf.item():.2%}")
        print(f"foreground:      {100.0 * _mask.mean():.1f}% of pixels")
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(_img); axes[0].set_title("Input image"); axes[0].axis("off")
        _arr = np.asarray(_img, dtype=np.float32) / 255.0
        _arr = _arr.copy(); _arr[_mask > 0] = 0.5 * _arr[_mask > 0] + 0.5 * np.array([1.0, 0.0, 0.0])
        axes[1].imshow(np.clip(_arr, 0, 1))
        axes[1].set_title(f"Prediction: {_CLASSES[int(_lab)]} ({_conf.item():.0%})"); axes[1].axis("off")
        plt.tight_layout(); plt.show()


## 8. Bonus — classifier backbone comparison

ResNet-18, MobileNetV3-Small and EfficientNet-B0, each trained for 10 epochs on
the same split. Test metrics below (loads from data/bonus/ if present).


In [ ]:
bonus_rows = []
for bb in ["resnet18", "mobilenet_v3_small", "efficientnet_b0"]:
    p = os.path.join(DATA_DIR, "bonus", bb, "results.json")
    if os.path.exists(p):
        r = json.load(open(p))["test"]["cls"]
        bonus_rows.append({"backbone": bb, "acc": r["acc"], "precision": r["precision"],
                           "recall": r["recall"], "f1": r["f1"]})
    else:
        bonus_rows.append({"backbone": bb, "acc": None, "precision": None,
                           "recall": None, "f1": None})
bonus_df = pd.DataFrame(bonus_rows).set_index("backbone")
display(bonus_df.round(4))
if bonus_df["acc"].notna().any():
    print("\nBest backbone by test accuracy:", bonus_df["acc"].idxmax())
else:
    print("\n(No per-backbone results found under data/bonus/ - copy the bonus "
          "results.json files there to show the comparison.)")


## Running this on a fresh machine

```bash
pip install torch torchvision pillow numpy matplotlib pandas ipywidgets
jupyter notebook          # open cse428_project.ipynb
```

- **present** mode: set `MODE = "present"` and run all cells — needs only the
  `data/` folder with the saved artifacts.
- **train** mode: set `MODE = "train"` and run all cells — needs the dataset
  (auto-downloads) and a GPU; 80 epochs takes a few hours.
